# Code Bases KEM Systems

### McEliece with BCH as a proof of concept, not safe for production use, goppa codes are not implemented as the decoder implementation in sagemath is using exhaustive search so testing is not feasible. 


In [26]:
# Import necessary libraries from SageMath and Python standard library
from sage.all import *
import hashlib
import os
field_extension_degree: int
error_correction_capability: int
class McElieceBCH_KEM:

    def __init__(self, field_extension_degree, error_correction_capability):
        """
        Initialize the McEliece KEM with BCH parameters.
        
        Args:
            field_extension_degree (int): The degree of the field extension (math: m).
            Defines code length n = 2^m - 1.
            error_correction_capability (int): The number of errors the code can correct (math: t).
        """
        self.field_extension_degree = field_extension_degree
        self.error_correction_capability = error_correction_capability
        
        # Calculate code length (n = 2^m - 1)
        self.code_length = 2**self.field_extension_degree - 1
        
        # Define the binary field GF(2)
        self.binary_field = GF(2)
        
        # Initialize the BCH Code
        # Designed distance is usually 2*t + 1
        designed_distance = 2 * self.error_correction_capability + 1
        self.bch_code = codes.BCHCode(self.binary_field, self.code_length, designed_distance)
        
        # Extract code dimension (k) and the Generator Matrix (G)
        self.code_dimension = self.bch_code.dimension()
        self.generator_matrix = self.bch_code.generator_matrix()
        
        # Placeholders for key dictionaries
        self.public_key = None
        self.private_key = None
        '''
        print(f"Variables:")
        print(f"  - Field Extension (m): {self.field_extension_degree}")
        print(f"  - Code Length (n):     {self.code_length}")
        print(f"  - Code Dimension (k):  {self.code_dimension}")
        print(f"  - Error Capacity (t):  {self.error_correction_capability}")
        '''

    def _generate_permutation_matrix(self, size):
        """Generates a random size x size permutation matrix (math: P)."""
        perm_matrix = Matrix(self.binary_field, size, size)
        permutation_list = list(range(size))
        shuffle(permutation_list)
        
        for row_index, col_index in enumerate(permutation_list):
            perm_matrix[row_index, col_index] = 1
        return perm_matrix

    def _generate_scrambler_matrix(self, size):
        """Generates a random size x size non-singular matrix (math: S)."""
        while True:
            matrix_candidate = Matrix.random(self.binary_field, size, size)
            if not matrix_candidate.is_singular():
                return matrix_candidate

    def generate_keypair(self):
        """
        Generates the Public and Private keys.
        
        Steps:
        1. Create a random k*k invertible Scrambler Matrix (S).
        2. Create a random n*n Permutation Matrix (P).
        3. Compute Public Key Matrix = S * G * P.
        """
       # print("Generating keypair...")
        
        # 1. Generate Scrambler Matrix S (k x k)
        scrambler_matrix = self._generate_scrambler_matrix(self.code_dimension)
        
        # 2. Generate Permutation Matrix P (n x n)
        permutation_matrix = self._generate_permutation_matrix(self.code_length)
        
        # 3. Compute Public Key: G_hat = S * G * P
        # Note: Sage operations are left-to-right: (S * G) * P
        public_key_matrix = scrambler_matrix * self.generator_matrix * permutation_matrix
        
        # Store Private Key Components
        self.private_key = {
            'scrambler_matrix': scrambler_matrix,
            'generator_matrix': self.generator_matrix,
            'permutation_matrix': permutation_matrix,
            'scrambler_inverse': scrambler_matrix.inverse(),
            'permutation_inverse': permutation_matrix.inverse()
        }
        
        # Store Public Key Components
        self.public_key = {
            'public_key_matrix': public_key_matrix,
            'error_capacity': self.error_correction_capability
        }
        
        #print("Keypair generated.")
        return self.public_key

    def encapsulate(self):
        """
        KEM Encapsulation (Alice).
        
        1. Create random message vector.
        2. Create random error vector.
        3. Encrypt.
        4. Hash message to get shared secret.
        """
        if self.public_key is None:
            raise RuntimeError("Keys not generated. Run generate_keypair() first.")
            
        public_matrix = self.public_key['public_key_matrix']
        max_errors = self.public_key['error_capacity']
        
        # 1. Generate random message vector (length k)
        random_message_vector = VectorSpace(self.binary_field, self.code_dimension).random_element()
        
        # 2. Generate random error vector (length n) with weight exactly t
        error_vector = vector(self.binary_field, self.code_length * [0])
        error_indices = sample(range(self.code_length), max_errors)
        for index in error_indices:
            error_vector[index] = 1
            
        # 3. Encrypt: ciphertext = message * G_pub + error
        ciphertext = random_message_vector * public_matrix + error_vector
        
        # 4. Derive Shared Secret (Hash of the random message)
        # Convert vector to bytes for hashing
        message_bytes = str(list(random_message_vector)).encode('utf-8')
        shared_secret = hashlib.sha256(message_bytes).hexdigest()
        
        return ciphertext, shared_secret

    def decapsulate(self, ciphertext):
        """
        KEM Decapsulation (Bob).
        
        1. Un-permute the ciphertext.
        2. Decode the code to remove errors.
        3. Recover the scrambled message.
        4. Un-scramble to get original message.
        5. Hash message to recover shared secret.
        """
        if self.private_key is None:
            raise RuntimeError("Private key missing.")
            
        permutation_inverse = self.private_key['permutation_inverse']
        scrambler_inverse = self.private_key['scrambler_inverse']
        base_generator = self.private_key['generator_matrix']
        
        # 1. Reverse Permutation: c' = c * P^-1
        # This results in: c' = (m * S * G) + (error * P^-1)
        # Since P is a permutation, (error * P^-1) still has weight t.
        unpermuted_ciphertext = ciphertext * permutation_inverse
        
        # 2. Decode using the BCH decoder
        # This removes the error vector, leaving us with: codeword = m * S * G
        try:
            cleaned_codeword = self.bch_code.decode_to_code(unpermuted_ciphertext)
        except Exception:
            raise ValueError("Decapsulation failed: Error correction unable to decode.")

        # 3. Recover the scrambled message (m_prime = m * S)
        # We solve the linear system: x * G = cleaned_codeword
        try:
            scrambled_message = base_generator.solve_left(cleaned_codeword)
        except ValueError:
            raise ValueError("Decoded vector is not in the span of the generator matrix.")

        # 4. Recover original message (m = m_prime * S^-1)
        original_message_vector = scrambled_message * scrambler_inverse
        
        # 5. Derive Shared Secret
        message_bytes = str(list(original_message_vector)).encode('utf-8')
        shared_secret = hashlib.sha256(message_bytes).hexdigest()
        
        return shared_secret



In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def single_mceliece_bch_kem(field_extension_degree=13, error_correction_capability=128):
    # m=6 implies n=63. t=2 implies we can correct 2 bits.
    kem_system = McElieceBCH_KEM(field_extension_degree, error_correction_capability)

    # 2. Key Generation
    pub_key = kem_system.generate_keypair()

    # 3. Encapsulation (Alice creates ciphertext and secret)
    ciphertext, alice_secret_key = kem_system.encapsulate()

    #print(f"Alice's Secret: {alice_secret_key}")

    # 4. Decapsulation (Bob receives ciphertext, derives secret)
    try:
        bob_secret_key = kem_system.decapsulate(ciphertext)
        #print(f"Bob's Secret:   {bob_secret_key}")

        # 5. Validation
        if alice_secret_key == bob_secret_key:
            #print("\n[SUCCESS] Shared secrets match.")
            return alice_secret_key, bob_secret_key, "SUCCESS"
        else:
            #print("\n[FAILURE] Secrets do not match.")
            return alice_secret_key, bob_secret_key, "FAILURE"

    except ValueError as error:
        print(f"\n[ERROR] {error}")



results = []

for _ in range(11):
    alice_secret_key, bob_secret_key, outcome = single_mceliece_bch_kem(field_extension_degree=6, error_correction_capability=2)
    results.append({'alice_shared_secret': alice_secret_key, 'bob_shared_secret': bob_secret_key, 'outcome': outcome})

pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(results)
df

,alice_shared_secret,bob_shared_secret,outcome
0,f30f00ced8b48c94565b2a607157c58feb6bd8dd3ed456b04ae42b03d05ad144,f30f00ced8b48c94565b2a607157c58feb6bd8dd3ed456b04ae42b03d05ad144,SUCCESS
1,48d14d0af8ae803d1101d94f68fa9d1b2c0dea2a465219ad1d7e8c33b8a9f20c,48d14d0af8ae803d1101d94f68fa9d1b2c0dea2a465219ad1d7e8c33b8a9f20c,SUCCESS
2,5aa1b5a2af88a5ab2a0306f67780fb1e6bae43b5cc2f56a2b28d265e7b782b41,5aa1b5a2af88a5ab2a0306f67780fb1e6bae43b5cc2f56a2b28d265e7b782b41,SUCCESS
3,4cb801c8e613b8094ce4819dd0de626dcda8bb84a8aacc51b642db1e912855bd,4cb801c8e613b8094ce4819dd0de626dcda8bb84a8aacc51b642db1e912855bd,SUCCESS
4,d60cdfc405b4b135128039420f53fa914cb8485787314a0d4363c34c7b5d99f1,d60cdfc405b4b135128039420f53fa914cb8485787314a0d4363c34c7b5d99f1,SUCCESS
5,8e79fbb8602fcc24493b9f4941189dd4eac46a07d90179c81296d878b15af0e6,8e79fbb8602fcc24493b9f4941189dd4eac46a07d90179c81296d878b15af0e6,SUCCESS
6,78404d8b6036b9d234d84bbcae9c5812a3c37c00e626a86faadeb92263f7e046,78404d8b6036b9d234d84bbcae9c5812a3c37c00e626a86faadeb92263f7e046,SUCCESS
7,0ea8376b6128c6a155d8e8562f47bfb91bf0f57a33fc05ec5bb5706d7218078c,0ea8376b6128c6a155d8e8562f47bfb91bf0f57a33fc05ec5bb5706d7218078c,SUCCESS
8,fee8f062dd2b8d255926ae9d9e3515c35b091f56be54341a450a5f8ba55707c6,fee8f062dd2b8d255926ae9d9e3515c35b091f56be54341a450a5f8ba55707c6,SUCCESS
9,a3489daffc3a429fe1808275672ff1c7f2a134140712412dcd2250a285e46a69,a3489daffc3a429fe1808275672ff1c7f2a134140712412dcd2250a285e46a69,SUCCESS


In [36]:
from sage.all import *
import hashlib
import random

class BikeKEM:
    def __init__(self, block_size=239, weight_density=38, error_weight=6):
        """
        Parameters for Toy/Demo version (Not secure for production):
        block_size (r): 239 (Prime)
        weight_density (w): 38 (Must be even so h0/h1 are odd)
        error_weight (t): 14
        """
        self.r = block_size
        self.w = weight_density
        self.t = error_weight
        
        # Define the Polynomial Ring R = F2[x] / (x^r - 1)
        # This handles all cyclic shifts and GF(2) arithmetic automatically.
        self.P = PolynomialRing(GF(2), 'x')
        self.R = self.P.quotient(self.P.gen() ** self.r - 1)
        self.x = self.P.gen()

    def _random_poly_of_weight(self, weight):
        """Generates a random polynomial with exactly 'weight' non-zero coefficients."""
        indices = random.sample(range(self.r), weight)
        poly = self.R(0)
        # We construct it this way to ensure it lives in the Quotient Ring
        raw_poly = sum(self.x**i for i in indices)
        return self.R(raw_poly)

    def _poly_to_indices(self, poly):
        """Helper: Get list of exponents where coefficient is 1."""
        # Lift to normal polynomial to inspect coefficients
        p_lift = poly.lift() 
        return [i for i, coeff in enumerate(p_lift.list()) if coeff == 1]

    def key_generation(self):
        while True:
            # 1. Generate h0, h1 with ODD weight.
            # Even weight polynomials are divisible by (x-1) and never invertible in this ring.
            w_half = self.w // 2
            # Ensure odd weight
            w0 = w_half if w_half % 2 == 1 else w_half + 1
            w1 = self.w - w0
            
            h0 = self._random_poly_of_weight(w0)
            h1 = self._random_poly_of_weight(w1)

            # 2. Check invertibility of h0
            if h0.is_unit():
                # Public key h = h1 * h0^(-1)
                self.private_key = {'h0': h0, 'h1': h1}
                self.public_key = h1 * h0.inverse()
                break
        return self.public_key

    def encapsulate(self):
        h = self.public_key
        
        # 1. Generate Error vectors e0, e1
        e0 = self._random_poly_of_weight(self.t // 2)
        e1 = self._random_poly_of_weight(self.t - (self.t // 2))
        
        # 2. Ciphertext c = e0 + e1 * h
        c = e0 + (e1 * h)
        
        # 3. Shared Secret K = Hash(e0, e1)
        secret_bytes = str(self._poly_to_indices(e0)).encode() + \
                       str(self._poly_to_indices(e1)).encode()
        shared_secret = hashlib.sha256(secret_bytes).hexdigest()
        
        return c, shared_secret

    def decapsulate(self, c):
        h0 = self.private_key['h0']
        h1 = self.private_key['h1']
        
        # 1. Compute initial Syndrome S = c * h0
        # This effectively equals: e0*h0 + e1*h1
        syndrome = c * h0
        
        e0_est = self.R(0)
        e1_est = self.R(0)
        
        # Pre-compute index lists for fast correlation checks
        # In a cyclic code, checking correlation with h0 means checking 
        # how many bits align between Syndrome and Rotations of h0.
        h0_indices = self._poly_to_indices(h0)
        h1_indices = self._poly_to_indices(h1)
        
        # --- SERIAL BIT-FLIPPING DECODER ---
        # We iterate, finding the ONE bit that reduces the syndrome weight the most,
        # flip it, and repeat. This prevents the "oscillation" seen in parallel versions.
        
        max_iter = self.r * 2  # Safety limit
        
        for _ in range(max_iter):
            s_indices = set(self._poly_to_indices(syndrome))
            s_weight = len(s_indices)
            
            if s_weight == 0:
                break
                
            best_flip = None
            max_correlation = -1
            target_poly = None # Tracks if we flip e0 or e1
            
            # --- Check Candidates in e0 ---
            # For each position i in block size, calculate UPC (Unsatisfied Parity Checks)
            # UPC_0[i] = | Syndrome AND (h0 shifted by i) |
            for i in range(self.r):
                # Calculate overlap between syndrome and h0 rotated by i
                # Note: In poly ring, rotation by i is multiplication by x^i
                # Optimization: We just add i to the indices mod r
                matches = 0
                for h_idx in h0_indices:
                    if (h_idx + i) % self.r in s_indices:
                        matches += 1
                
                if matches > max_correlation:
                    max_correlation = matches
                    best_flip = (0, i) # (vector_index, bit_index)

            # --- Check Candidates in e1 ---
            for i in range(self.r):
                matches = 0
                for h_idx in h1_indices:
                    if (h_idx + i) % self.r in s_indices:
                        matches += 1
                
                if matches > max_correlation:
                    max_correlation = matches
                    best_flip = (1, i)

            # --- Threshold Check ---
            # If the best correlation is small, we are just flipping noise. Stop.
            # Heuristic: We usually expect correlation > w/4 at least.
            if max_correlation <= (self.w // 4):
                break
                
            # --- Apply Flip ---
            vec_idx, bit_idx = best_flip
            flip_poly = self.R(self.x ** bit_idx)
            
            if vec_idx == 0:
                e0_est += flip_poly
                # Update syndrome: S_new = S_old + h0 * x^i
                syndrome += (h0 * flip_poly)
            else:
                e1_est += flip_poly
                # Update syndrome: S_new = S_old + h1 * x^i
                syndrome += (h1 * flip_poly)

        # Reconstruct Secret
        secret_bytes = str(self._poly_to_indices(e0_est)).encode() + \
                       str(self._poly_to_indices(e1_est)).encode()
        return hashlib.sha256(secret_bytes).hexdigest()

# --- VALIDATION ---
bike = BikeKEM()
bike.key_generation()

success = 0
trials = 50

print(f"Running {trials} trials with Serial Decoder...")
for i in range(trials):
    c, alice_s = bike.encapsulate()
    bob_s = bike.decapsulate(c)
    if alice_s == bob_s:
        success += 1

print(f"Success rate: {success}/{trials}")
if success == trials:
    print("Perfect decoding!")
else:
    print("Partial success - adjust 'error_weight' down if failures persist.")

Running 50 trials with Serial Decoder...
Success rate: 50/50
Perfect decoding!


In [37]:
import time

def run_isd_attack(bike_instance, ciphertext_poly):
    """
    Performs Prange's ISD attack to recover e0 and e1.
    """
    r = bike_instance.r
    t = bike_instance.t
    n = 2 * r
    h_poly = bike_instance.public_key
    
    print(f"[*] Starting Prange ISD (r={r}, t={t})")
    print(f"[*] Probability of success per iteration: ~1/{int(1/((binomial(r, t)/binomial(n, t)))):,}")
    
    # 1. Construct the Parity Check Matrix H = [I | Rot(h)^T]
    # In BIKE, c = e0 + e1*h => e0 + e1*h - c = 0
    # This is equivalent to H * [e0, e1]^T = c^T
    
    # Create the circulant matrix for h
    h_coeffs = h_poly.lift().list()
    h_coeffs += [0] * (r - len(h_coeffs))
    
    # Rows of Rot(h) are cyclic shifts
    rot_h_rows = [h_coeffs[-i:] + h_coeffs[:-i] for i in range(r)]
    Rot_h = matrix(GF(2), rot_h_rows)
    
    # H = [I | Rot(h).transpose()]
    H = block_matrix(GF(2), [[identity_matrix(r), Rot_h.transpose()]])
    
    # Target syndrome is the ciphertext vector
    s = vector(GF(2), ciphertext_poly.lift().list() + [0]*(r - len(ciphertext_poly.lift().list())))
    
    start_time = time.time()
    iterations = 0
    
    while True:
        iterations += 1
        if iterations % 100 == 0:
            elapsed = time.time() - start_time
            print(f"    Iter {iterations}... ({elapsed:.2f}s)", end='\r')
            
        # 2. Pick a random Information Set (r columns from 2r)
        cols = random.sample(range(n), r)
        H_sub = H[:, cols]
        
        # 3. Try to solve H_sub * e_sub = s
        try:
            # solve_right finds x such that H_sub * x = s
            e_sub = H_sub.solve_right(s)
        except ValueError:
            # Matrix was not invertible, skip
            continue
            
        # 4. Check if the weight of the solution matches our target t
        # (Prange assumes all errors are trapped in the chosen information set)
        if e_sub.hamming_weight() == t:
            end_time = time.time()
            print(f"\n[+] SUCCESS! Found error vector after {iterations} iterations.")
            print(f"[+] Total time: {end_time - start_time:.2f} seconds")
            
            # Reconstruct the full 2r vector
            e_full = vector(GF(2), n)
            for i, col_idx in enumerate(cols):
                e_full[col_idx] = e_sub[i]
                
            # Split into e0 and e1
            e0_recovered = bike_instance.R(list(e_full[:r]))
            e1_recovered = bike_instance.R(list(e_full[r:]))
            
            return e0_recovered, e1_recovered

# --- Execution ---
# Note: I'm using t=6 as in your __init__ default for a fast demo.
# If t=14, this could take a significantly longer time.
bike = BikeKEM(block_size=239, weight_density=38, error_weight=6)
pk = bike.key_generation()
c, secret_alice = bike.encapsulate()

# Run the attack
e0_atk, e1_atk = run_isd_attack(bike, c)

# Verify against actual secrets (accessing private members for validation)
# Note: In your class, e0/e1 aren't stored, but we can check if they decrypt
# to the same shared secret.
secret_atk_bytes = str(bike._poly_to_indices(e0_atk)).encode() + \
                   str(bike._poly_to_indices(e1_atk)).encode()
secret_atk = hashlib.sha256(secret_atk_bytes).hexdigest()

print("\n--- RESULTS ---")
print(f"Original Secret:  {secret_alice}")
print(f"Attacker Secret:  {secret_atk}")
print(f"Match:            {secret_alice == secret_atk}")

[*] Starting Prange ISD (r=239, t=6)
[*] Probability of success per iteration: ~1/66

[+] SUCCESS! Found error vector after 21 iterations.
[+] Total time: 0.03 seconds

--- RESULTS ---
Original Secret:  2d7c878fd3fca71c3708e080c2607a6627108b0625681271b23704516eb2a0d2
Attacker Secret:  2d7c878fd3fca71c3708e080c2607a6627108b0625681271b23704516eb2a0d2
Match:            True


In [41]:
import pandas as pd

def bike_t_stress_test(block_size=239, weight_density=38, t_range=range(4, 21, 2), trials=20):
    results = []
    
    for t in t_range:
        successes = 0
        bike = BikeKEM(block_size=block_size, weight_density=weight_density, error_weight=t)
        
        for _ in range(trials):
            bike.key_generation()
            c, alice_s = bike.encapsulate()
            bob_s = bike.decapsulate(c)
            if alice_s == bob_s:
                successes += 1
        
        results.append({
            "t (Error Weight)": t,
            "Success Rate (%)": float((successes / trials) * 100),
            "Block Size (r)": block_size,
            "Density (w)": weight_density
        })
    
    return pd.DataFrame(results)

# Run and display
df_t_test = bike_t_stress_test()
display(df_t_test)

,t (Error Weight),Success Rate (%),Block Size (r),Density (w)
0,4,100.0,239,38
1,6,100.0,239,38
2,8,95.0,239,38
3,10,75.0,239,38
4,12,15.0,239,38
5,14,0.0,239,38
6,16,0.0,239,38
7,18,0.0,239,38
8,20,0.0,239,38


In [40]:
def bike_scaling_test(fixed_t=6, r_range=[101, 151, 239, 317, 401], trials=20):
    results = []
    
    for r in r_range:
        # We scale density w relative to r (approx sqrt(r))
        w = int(sqrt(r) * 2.5) 
        if w % 2 != 0: w += 1 # Ensure even for h0/h1 odd splits
        
        successes = 0
        bike = BikeKEM(block_size=r, weight_density=w, error_weight=fixed_t)
        
        for _ in range(trials):
            bike.key_generation()
            c, alice_s = bike.encapsulate()
            bob_s = bike.decapsulate(c)
            if alice_s == bob_s:
                successes += 1
                
        results.append({
            "Block Size (r)": r,
            "Weight (w)": w,
            "Success Rate (%)": float((successes / trials) * 100),
            "Fixed t": fixed_t
        })
        
    return pd.DataFrame(results)

df_scaling = bike_scaling_test()
display(df_scaling)

,Block Size (r),Weight (w),Success Rate (%),Fixed t
0,101,26,40.0,6
1,151,30,100.0,6
2,239,38,100.0,6
3,317,44,100.0,6
4,401,50,100.0,6


In [43]:
import time
import pandas as pd

def prange_isd_mceliece(public_matrix, ciphertext, t_weight):
    """
    Prange's ISD Attack tailored for the provided McElieceBCH_KEM.
    Target: Find error vector 'e' such that c = m*G_pub + e
    """
    k = public_matrix.nrows()
    n = public_matrix.ncols()
    field = public_matrix.base_ring()
    
    # Target: c = m*G_pub + e
    # If we pick k columns where e is 0, then c_sub = m * G_pub_sub
    # Therefore m = c_sub * (G_pub_sub)^-1
    
    start_time = time.time()
    iterations = 0
    
    while True:
        iterations += 1
        
        # 1. Select a random Information Set (k columns)
        cols = random.sample(range(n), k)
        G_sub = public_matrix[:, cols]
        
        # 2. Check if submatrix is invertible
        if not G_sub.is_singular():
            # 3. Solve for potential message m_cand
            c_sub = vector(field, [ciphertext[i] for i in cols])
            m_cand = G_sub.solve_left(c_sub)
            
            # 4. Reconstruct the full codeword and find the error
            e_cand = ciphertext - (m_cand * public_matrix)
            
            # 5. Check if the Hamming weight matches t
            if e_cand.hamming_weight() == t_weight:
                elapsed = time.time() - start_time
                return m_cand, e_cand, iterations, elapsed

def run_mceliece_attack_harness(m_param=6, t_param=2, trials=5):
    """
    Harness to run trials of the ISD attack against the BCH McEliece system.
    """
    attack_results = []
    
    # Initialize the system
    kem = McElieceBCH_KEM(field_extension_degree=m_param, error_correction_capability=t_param)
    
    print(f"[*] Starting Attack Harness: n={kem.code_length}, k={kem.code_dimension}, t={t_param}")
    
    for i in range(trials):
        # Setup specific trial
        pk = kem.generate_keypair()
        c, true_secret = kem.encapsulate()
        
        # Run Attack
        m_found, e_found, iters, duration = prange_isd_mceliece(
            pk['public_key_matrix'], 
            c, 
            pk['error_capacity']
        )
        
        # Verify Shared Secret recovery
        # Replicating your internal hashing logic
        message_bytes = str(list(m_found)).encode('utf-8')
        recovered_secret = hashlib.sha256(message_bytes).hexdigest()
        
        attack_results.append({
            "Trial": i + 1,
            "Iterations": iters,
            "Time (s)": round(duration, 4),
            "Secret Match": (true_secret == recovered_secret),
            "Weight Found": e_found.hamming_weight()
        })
        
    return pd.DataFrame(attack_results)

# --- EXECUTION ---
# Using your suggested values: m=6 (n=63), t=2
df_attack = run_mceliece_attack_harness(m_param=6, t_param=2, trials=10)
display(df_attack)

[*] Starting Attack Harness: n=63, k=51, t=2


,Trial,Iterations,Time (s),Secret Match,Weight Found
0,1,1,0.0008,True,2
1,2,13,0.0037,True,2
2,3,90,0.0218,True,2
3,4,142,0.0273,True,2
4,5,143,0.0247,True,2
5,6,1,0.0005,True,2
6,7,94,0.0149,True,2
7,8,27,0.0051,True,2
8,9,64,0.0113,True,2
9,10,34,0.0066,True,2
